In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Config
TARGET = 'Subscribed'
ID_COL = 'id'
FOLDS = 5

CATBOOST_SEEDS = [42, 2024, 3407]
NN_SEEDS = [42, 2024, 3407, 777, 1337, 1001, 2718]

MONTH_MAP = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}

CB_PARAMS = {
    'iterations': 3500, 'learning_rate': 0.01377, 'depth': 6,
    'l2_leaf_reg': 3.976, 'random_strength': 1.084, 'bagging_temperature': 0.733,
    'bootstrap_type': 'Bayesian', 'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'allow_writing_files': False, 'verbose': False, 'thread_count': -1
}

NN_PARAMS = {'emb_dim': 24, 'hidden_dim': 192, 'n_layers': 4, 'heads': 6,
             'dropout': 0.2, 'epochs': 90, 'batch_size': 1024, 'lr': 0.001,
             'weight_decay': 0.01, 'patience': 16}

BASE_CAT = ['job','marital','education','default','housing','loan','contact','month','poutcome']

def rank_norm(v): return pd.Series(v).rank(method='average', pct=True).to_numpy()
def _s(d,c,l=False): v=d[c].fillna('missing').astype(str); return v.str.lower() if l else v

print('Setup complete')

In [ ]:
def build_features(data):
    df = data.copy()
    if ID_COL in df.columns: df = df.drop(columns=[ID_COL])
    
    m,j,ma,e,ct,po,l,d,h = [_s(df,c,c=='month') for c in ['month','job','marital','education','contact','poutcome','loan','default','housing']]
    
    df['month'] = m
    df['month_num'] = m.map(MONTH_MAP).fillna(0).astype(np.int16)
    df['pdays_was_missing'] = (df['pdays']==-1).astype(np.int8)
    df['pdays_clean'] = df['pdays'].replace(-1, 999)
    
    # Log transforms
    for col in ['duration','balance','campaign','previous']:
        df[f'{col}_log1p'] = np.log1p(df[col].clip(lower=0))
    df['balance_abs_log1p'] = np.log1p(df['balance'].abs())
    df['pdays_log1p'] = np.log1p(df['pdays_clean'])
    
    # Interactions
    df['contacts_total'] = df['campaign'] + df['previous']
    df['duration_per_campaign'] = df['duration'] / (df['campaign']+1)
    df['balance_per_age'] = df['balance'] / (df['age']+1)
    df['previous_per_campaign'] = df['previous'] / (df['campaign']+1)
    df['campaign_x_previous'] = df['campaign'] * df['previous']
    df['duration_x_campaign'] = df['duration'] * df['campaign']
    
    # Binary
    df['has_any_loan'] = ((h=='yes')|(l=='yes')).astype(np.int8)
    df['is_default'] = (d=='yes').astype(np.int8)
    df['was_contacted_before'] = (df['pdays']!=-1).astype(np.int8)
    df['is_cellular'] = (ct=='cellular').astype(np.int8)
    df['balance_signed_log1p'] = np.sign(df['balance']) * np.log1p(df['balance'].abs())
    df['balance_negative'] = (df['balance']<0).astype(np.int8)
    df['balance_nonpositive'] = (df['balance']<=0).astype(np.int8)
    
    # Buckets
    ps = df['pdays'].replace(-1,999)
    df['age_bucket'] = pd.cut(df['age'],[0,25,35,45,55,65,120],labels=['<=25','26-35','36-45','46-55','56-65','65+']).astype(str)
    df['campaign_bucket'] = pd.cut(df['campaign'],[-1,1,2,4,9,np.inf],labels=['1','2','3-4','5-9','10+']).astype(str)
    df['previous_bucket'] = pd.cut(df['previous'],[-1,0,1,3,np.inf],labels=['0','1','2-3','4+']).astype(str)
    df['pdays_bucket'] = pd.cut(ps,[-1,7,30,90,365,np.inf],labels=['<=1w','8-30d','31-90d','91-365d','365d+']).astype(str)
    df.loc[df['pdays']==-1,'pdays_bucket'] = 'never'
    df['duration_bucket'] = pd.cut(df['duration'],[-1,60,120,240,480,np.inf],labels=['<=1m','1-2m','2-4m','4-8m','8m+']).astype(str)
    df['day_bucket'] = pd.cut(df['day'],[0,10,20,31],labels=['early','mid','late'],include_lowest=True).astype(str)
    
    # Crosses
    df['job_education'] = j+'__'+e
    df['job_marital'] = j+'__'+ma
    df['contact_month'] = ct+'__'+m
    df['poutcome_month'] = po+'__'+m
    df['loan_default'] = l+'__'+d
    df['contact_day_bucket'] = ct+'__'+df['day_bucket']
    df['month_day_bucket'] = m+'__'+df['day_bucket']
    df['history_state'] = np.where(df['previous']>0, po+'__seen', 'no_previous')
    
    cats = BASE_CAT + ['age_bucket','campaign_bucket','previous_bucket','pdays_bucket',
        'duration_bucket','day_bucket','job_education','job_marital','contact_month',
        'poutcome_month','loan_default','contact_day_bucket','month_day_bucket','history_state']
    for c in cats: df[c] = df[c].fillna('missing').astype(str)
    return df, cats

print('Features defined')

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, dropout, ff_mult=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, dim*ff_mult), nn.GELU(), nn.Dropout(dropout),
                                nn.Linear(dim*ff_mult, dim), nn.Dropout(dropout))
    def forward(self, x):
        a = self.norm1(x)
        o, _ = self.attn(a, a, a, need_weights=False)
        x = x + o
        return x + self.ff(self.norm2(x))

class AttentionNet(nn.Module):
    def __init__(self, num_numeric, cat_dims, emb_dim, hidden_dim, n_layers, heads, dropout):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(d, emb_dim) for d in cat_dims])
        self.num_proj = nn.Linear(num_numeric, hidden_dim) if num_numeric > 0 else None
        self.cat_proj = nn.Linear(emb_dim, hidden_dim) if cat_dims else None
        self.cls_token = nn.Parameter(torch.randn(1,1,hidden_dim)*0.02)
        self.blocks = nn.ModuleList([TransformerBlock(hidden_dim, heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, hidden_dim//2),
                                  nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
    def forward(self, x_num, x_cat):
        tokens = []
        if self.num_proj is not None and x_num.shape[1] > 0:
            tokens.append(self.num_proj(x_num).unsqueeze(1))
        if self.embeddings:
            for i, emb in enumerate(self.embeddings):
                tokens.append(self.cat_proj(emb(x_cat[:,i])).unsqueeze(1))
        bs = x_num.shape[0]
        tokens.insert(0, self.cls_token.expand(bs,-1,-1))
        x = torch.cat(tokens, dim=1)
        for block in self.blocks: x = block(x)
        return torch.sigmoid(self.head(x[:,0]))

print('Models defined')

In [ ]:
def train_catboost(x_tr, x_te, y, cats, params, seeds):
    oof, test = np.zeros(len(x_tr)), np.zeros(len(x_te))
    for seed in seeds:
        s_oof, s_test = np.zeros(len(x_tr)), np.zeros(len(x_te))
        for fi, (ti, vi) in enumerate(StratifiedKFold(FOLDS, True, seed).split(x_tr, y), 1):
            m = CatBoostClassifier(**params, auto_class_weights='Balanced', random_seed=seed)
            m.fit(x_tr.iloc[ti].reset_index(drop=True), y.iloc[ti].reset_index(drop=True),
                  eval_set=(x_tr.iloc[vi].reset_index(drop=True), y.iloc[vi].reset_index(drop=True)),
                  cat_features=cats, use_best_model=True, early_stopping_rounds=250, verbose=False)
            s_oof[vi] = m.predict_proba(x_tr.iloc[vi].reset_index(drop=True))[:,1]
            s_test += m.predict_proba(x_te)[:,1] / FOLDS
        print(f'CB seed {seed}: {roc_auc_score(y, s_oof):.6f}')
        oof += s_oof / len(seeds)
        test += s_test / len(seeds)
    print(f'CB Final: {roc_auc_score(y, oof):.6f}')
    return oof, test

def train_nn(x_tr, x_te, y, cats, params, seeds):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'NN on {device}')
    num_cols = [c for c in x_tr.columns if c not in cats]
    
    # Encode
    cat_dims, x_cat_tr, x_cat_te = [], np.zeros((len(x_tr),len(cats)),dtype=np.int64), np.zeros((len(x_te),len(cats)),dtype=np.int64)
    for i, c in enumerate(cats):
        comb = pd.concat([x_tr[c].fillna('missing').astype(str), x_te[c].fillna('missing').astype(str)], ignore_index=True)
        codes, _ = pd.factorize(comb, sort=True)
        x_cat_tr[:,i], x_cat_te[:,i] = codes[:len(x_tr)], codes[len(x_tr):]
        cat_dims.append(int(codes.max()+1))
    
    scaler = StandardScaler()
    x_num_tr = scaler.fit_transform(x_tr[num_cols].fillna(x_tr[num_cols].median()).astype(np.float32)) if num_cols else np.zeros((len(x_tr),0),dtype=np.float32)
    x_num_te = scaler.transform(x_te[num_cols].fillna(x_tr[num_cols].median()).astype(np.float32)) if num_cols else np.zeros((len(x_te),0),dtype=np.float32)
    
    oof, test = np.zeros(len(x_tr)), np.zeros(len(x_te))
    y_np = y.values.astype(np.float32)
    
    for seed in seeds:
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        s_oof, s_test = np.zeros(len(x_tr)), np.zeros(len(x_te))
        
        for fi, (ti, vi) in enumerate(StratifiedKFold(FOLDS, True, seed).split(x_tr, y), 1):
            tr_num, tr_cat = torch.FloatTensor(x_num_tr[ti]).to(device), torch.LongTensor(x_cat_tr[ti]).to(device)
            tr_y = torch.FloatTensor(y_np[ti]).unsqueeze(1).to(device)
            va_num, va_cat = torch.FloatTensor(x_num_tr[vi]).to(device), torch.LongTensor(x_cat_tr[vi]).to(device)
            te_num, te_cat = torch.FloatTensor(x_num_te).to(device), torch.LongTensor(x_cat_te).to(device)
            
            model = AttentionNet(x_num_tr.shape[1], cat_dims, **{k:params[k] for k in ['emb_dim','hidden_dim','n_layers','heads','dropout']}).to(device)
            opt = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
            crit, sched = nn.BCELoss(), torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'max', patience=8, factor=0.5)
            loader = DataLoader(TensorDataset(tr_num, tr_cat, tr_y), batch_size=params['batch_size'], shuffle=True)
            
            amp = device.type == 'cuda'
            scaler_amp = torch.cuda.amp.GradScaler(enabled=amp)
            best_auc, best_va, best_te, patience = 0, None, None, 0
            
            for ep in range(params['epochs']):
                model.train()
                for bn, bc, by in loader:
                    opt.zero_grad()
                    with torch.cuda.amp.autocast(enabled=amp): loss = crit(model(bn, bc), by)
                    scaler_amp.scale(loss).backward()
                    scaler_amp.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler_amp.step(opt); scaler_amp.update()
                
                model.eval()
                with torch.no_grad():
                    with torch.cuda.amp.autocast(enabled=amp): va_pred = model(va_num, va_cat).cpu().numpy().flatten()
                    auc = roc_auc_score(y.iloc[vi], va_pred); sched.step(auc)
                    if auc > best_auc:
                        best_auc, best_va, patience = auc, va_pred.copy(), 0
                        with torch.cuda.amp.autocast(enabled=amp): best_te = model(te_num, te_cat).cpu().numpy().flatten()
                    else: patience += 1
                    if patience >= params['patience']: break
            
            s_oof[vi] = best_va
            s_test += best_te / FOLDS
        
        print(f'NN seed {seed}: {roc_auc_score(y, s_oof):.6f}')
        oof += s_oof / len(seeds)
        test += s_test / len(seeds)
    print(f'NN Final: {roc_auc_score(y, oof):.6f}')
    return oof, test

print('Training functions ready')

In [ ]:
# Load data
train = pd.read_csv('/kaggle/input/fiicode-2026-ai-competition/train.csv')
test = pd.read_csv('/kaggle/input/fiicode-2026-ai-competition/test.csv')
y = train[TARGET].astype(int)
print(f'Train: {len(train)}, Test: {len(test)}, Target: {y.mean():.4f}')

# Build features
x_train, cats = build_features(train.drop(columns=[TARGET]))
x_test, _ = build_features(test)
print(f'Features: {len(x_train.columns)}, Categorical: {len(cats)}')

# Train models
print('\n=== CatBoost ===')
cb_oof, cb_test = train_catboost(x_train, x_test, y, cats, CB_PARAMS, CATBOOST_SEEDS)

print('\n=== AttentionNN ===')
nn_oof, nn_test = train_nn(x_train, x_test, y, cats, NN_PARAMS, NN_SEEDS)

# Blend 80/20 (proven best on LB)
print('\n=== Blending 80/20 ===')
blend_oof = 0.80 * rank_norm(cb_oof) + 0.20 * rank_norm(nn_oof)
blend_test = 0.80 * rank_norm(cb_test) + 0.20 * rank_norm(nn_test)
print(f'Blend OOF AUC: {roc_auc_score(y, blend_oof):.6f}')

# Save submission
submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: np.clip(blend_test, 1e-6, 1-1e-6)})
submission.to_csv('submission.csv', index=False)
print('\nSaved submission.csv')
submission.head(10)